In [1]:
import pandas as pd
from pathlib import Path


data_dir = Path.cwd().parents[1] / "data" / "raw"


df_clientes = pd.read_csv(data_dir / "clientes.csv", encoding="utf-8")
df_compras = pd.read_csv(data_dir / "compras.csv", sep=",", encoding="utf-8")
df_productos = pd.read_csv(data_dir / "productos.csv", sep=",", encoding="utf-8")
df_ventas = pd.read_csv(data_dir / "ventas.csv", sep=",", encoding="utf-8")


#Creamos los csv de rechazados para clientes, compras, productos y ventas

df_rechazados_clientes = pd.DataFrame(
    columns=[*df_clientes.columns, "motivo_rechazo"]
)
df_rechazados_compras = pd.DataFrame(
    columns=[*df_compras.columns, "motivo_rechazo"]
)
df_rechazados_productos = pd.DataFrame(
    columns=[*df_productos.columns, "motivo_rechazo"]
)
df_rechazados_ventas = pd.DataFrame(
    columns=[*df_ventas.columns, "motivo_rechazo"]
)


In [ ]:
#Visualizamos cada uno de los csv
#df_clientes.head()

#df_clientes.info()
#Los tipos de datos de las columnas estan bien asi que proseguimos con la etapa de validacion



#Buscamos valores nulos en todas las columnas
#print(df_clientes.isnull().sum())
#Hay 2 valores nulos en provincia (11 y 16) y 3 en email (35,42,54)

""" 
ETAPA DE VALIDACION DE DATOS
"""

#--------------------------------------------------------------------------#
# Filtra reales nulos (NaN/None) O strings vacíos/con solo espacios en pronvincia
condicion_provincias = df_clientes["provincia"].isnull() | (df_clientes["provincia"].astype(str).str.strip() == "")
df_clientes[condicion_provincias]


#---------------------------------------------------------------------------#
# Filtra reales nulos (NaN/None) O strings vacíos/con solo espacios en emails
condicion_emails = df_clientes["email"].isnull() | (df_clientes["email"].astype(str).str.strip() == "")
df_clientes[condicion_emails]


#Dejamos los registros con email nulo o vacio ya que no es un campo obligatorio, por lo que no los mandamos al csv de rechazados

#---------------------------------------------------------------------------#

#Buscamos valores duplicados en todas las columnas
duplicados = df_clientes[df_clientes.duplicated(keep=False)]
#print(duplicados)
#Estan duplicados 27, 31, 32
#El id_cliente 14 esta duplicado con otros datos y el 56 tambien esta duplicado con otros datos, por lo que los metemos al csv de rechazados

#--------------------------------------------------------------------------#
#Creamos las mascaras para los registros que tienen provincia vacia o nula y los registros duplicados
#Marca solo las copias exactas sobrantes; conserva la primera

mask_provincia_vacia = df_clientes["provincia"].isnull() | (df_clientes["provincia"].astype(str).str.strip() == "")
mask_copia_duplicada = df_clientes.duplicated(keep="first")

mask_mails_mal_formateados = ~df_clientes["email"].str.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", na=True)


# Encuentra IDs repetidos cuyas filas tienen datos distintos
duplicados_por_id = df_clientes[
    df_clientes.duplicated(subset=["id_cliente"], keep=False)
]
ids_conflictivos = (
    duplicados_por_id.groupby("id_cliente")
    .filter(lambda grupo: len(grupo.drop_duplicates()) > 1)["id_cliente"]
    .unique()
)
mask_id_conflictivo = df_clientes["id_cliente"].isin(ids_conflictivos)



#Construimos el csv de rechazados y validos con los registros que tienen provincia vacia o nula y los registros duplicados
mask_rechazo = (
    mask_provincia_vacia
    | mask_copia_duplicada
    | mask_id_conflictivo
    | mask_mails_mal_formateados
)
#Construimos los motivos de rechazo para los registros que tienen provincia vacia o nula y los registros duplicados
motivos = pd.Series("", index=df_clientes.index, dtype="string")

motivos.loc[mask_provincia_vacia] = (
    motivos.loc[mask_provincia_vacia] + "provincia_vacia; "
)
motivos.loc[mask_copia_duplicada] = (
    motivos.loc[mask_copia_duplicada] + "copia_exacta_duplicada; "
)
motivos.loc[mask_id_conflictivo] = (
    motivos.loc[mask_id_conflictivo] + "id_cliente_conflictivo; "
)
motivos.loc[mask_mails_mal_formateados] = (
    motivos.loc[mask_mails_mal_formateados] + "email_mal_formateado; "
)



df_rechazados_clientes = df_clientes.loc[mask_rechazo].copy()
df_clientes_validos = df_clientes.loc[~mask_rechazo].copy()

df_rechazados_clientes["motivo_rechazo"] = (
    motivos.loc[mask_rechazo].str.rstrip("; ")
)






    id_cliente                  nombre provincia   localidad  \
0            1     Autoservicio Romero  San Luis    San Luis   
5            6    Distribuidora García  San Juan   Rivadavia   
48          49  Autoservicio Rodríguez   Mendoza  Godoy Cruz   

                         email  
0       contacto1romero.com.ar  
5       contacto6garcia.com.ar  
48  contacto49rodriguez.com.ar  


In [9]:
df_compras.head()

""" 
ETAPA DE VALIDACION DE DATOS
"""

#Corregimos el tipo de dato precio unitario
df_compras["costo_unitario"] = df_compras["costo_unitario"].astype(float)

#---------------------------------------------------------------------------#

#Buscamos valores nulos en todas las columnas
#print(df_compras.isnull().sum())
#Hay 2 valores nulos en fecha (5088, 5275) y 4 en proveedor (5010, 5230, 5300, 5196)
# Filtra reales nulos (NaN/None) O strings vacíos/con solo espacios en fecha
condicion_fecha = df_compras["fecha"].isnull() | (df_compras["fecha"].astype(str).str.strip() == "")
df_compras[condicion_fecha]

#Revisamos si hay fechas futuras en la columna fecha y vemos cuantas son
raw = df_compras["fecha"].astype("string").str.strip()

fecha_convertida = pd.Series(
    pd.NaT,
    index=df_compras.index,
    dtype="datetime64[ns]"
)

formatos = [
    (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d"),  # 2026-11-12
    (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y"),  # 13/11/2026
    (r"\d{2}-\d{2}-\d{2}", "%d-%m-%y"),  # 09-10-26
]

for patron, formato in formatos:
    mascara = raw.str.fullmatch(patron, na=False)
    fecha_convertida.loc[mascara] = pd.to_datetime(
        raw.loc[mascara],
        format=formato,
        errors="coerce"
    )

#Sumamos el fecha_convertida a la columna fecha del df_compras
df_compras["fecha_convertida"] = fecha_convertida



# Filtra reales nulos (NaN/None) O strings vacíos/con solo espacios en proveedor
condicion_proveedor= df_compras["proveedor"].isnull() | (df_compras["proveedor"].astype(str).str.strip() == "")
#df_compras[condicion_proveedor]



#---------------------------------------------------------------------------#

#Buscamos valores duplicados en todas las columnas
#print(df_compras.duplicated().sum())
#Hay 5 valores duplicados (5020 , 5078 , 5104, 5105 , 5193)



#--------------------------------------------------------------------------#

#Buscamos los productos con precios menores o iguales a 0
df_compras_precio0 = df_compras[df_compras["costo_unitario"] <= 0]
#print(df_compras_precio0)
#Hay 4 productos con precio menor o igual a 0 (5262,5007,5258,5120)



#--------------------------------------------------------------------------#
#Buscamos los productos con cantidades menores o iguales a 0
df_compras_cantidad0 = df_compras[df_compras["cantidad"] <= 0]
#print(df_compras_cantidad0)
#Hay 5 productos con cantidades menores a 0 (5172,5247,5090,5035,5163)


#--------------------------------------------------------------------------#

#Creamos las mascaras para los registros que tienen fecha nula o vacia, proveedor nulo o vacio, precio menor o igual a 0, cantidad menor o igual a 0 y los registros duplicados
mask_fecha_nula = (
    df_compras["fecha"].isnull() | (df_compras["fecha"].astype(str).str.strip() == "") | df_compras['fecha_convertida'].isna()
)
mask_proveedor_nulo = df_compras["proveedor"].isnull() | (df_compras["proveedor"].astype(str).str.strip() == "")
mask_precio0 = df_compras["costo_unitario"] <= 0 | df_compras["costo_unitario"].isnull()
mask_cantidad0 = df_compras["cantidad"] <= 0 | df_compras["cantidad"].isnull()
mask_copia_duplicada = df_compras.duplicated(keep="first") | df_compras.duplicated(subset=["id_compra"], keep="first")
mask_id_duplicado_compras = df_compras.duplicated(subset=["id_compra"], keep="first")
mask_fecha_futura = df_compras["fecha_convertida"] > pd.Timestamp.now()
mask_id_producto_valido = df_compras["id_producto"].isin(df_productos["id_producto"])


mask_rechazo = (
    mask_fecha_nula
    | mask_proveedor_nulo
    | mask_precio0
    | mask_cantidad0
    | mask_copia_duplicada
    | mask_id_duplicado_compras
    | mask_fecha_futura
    | ~mask_id_producto_valido
)

motivos = pd.Series("", index=df_compras.index, dtype="string")
motivos.loc[mask_fecha_nula] = (
    motivos.loc[mask_fecha_nula] + "fecha_nula_o_mal_formateada; "
)
motivos.loc[mask_proveedor_nulo] = (
    motivos.loc[mask_proveedor_nulo] + "proveedor_nulo_o_vacio; "
)
motivos.loc[mask_precio0] = (
    motivos.loc[mask_precio0] + "precio_unitario_menor_o_igual_a_0; "
)
motivos.loc[mask_cantidad0] = (
    motivos.loc[mask_cantidad0] + "cantidad_menor_o_igual_a_0; "
)
motivos.loc[mask_copia_duplicada] = (
    motivos.loc[mask_copia_duplicada] + "copia_exacta_duplicada; "
)
motivos.loc[mask_id_duplicado_compras] = (
    motivos.loc[mask_id_duplicado_compras] + "id_compra_duplicado; "
)
motivos.loc[mask_fecha_futura] = (
    motivos.loc[mask_fecha_futura] + "fecha_futura; "
)
motivos.loc[~mask_id_producto_valido] = (
    motivos.loc[~mask_id_producto_valido] + "id_producto_invalido; "
)   

df_rechazados_compras = df_compras.loc[mask_rechazo].copy()
df_compras_validos = df_compras.loc[~mask_rechazo].copy()

df_rechazados_compras["motivo_rechazo"] = (
    motivos.loc[mask_rechazo].str.rstrip("; ")
)




In [ ]:
#df_productos.head()


#Corregimos el tipo de dato precio
df_productos["precio"] = df_productos["precio"].astype(float)
#df_productos.info()

#---------------------------------------------------------------------------#

#Buscamos valores nulos en todas las columnas de categoria
#print(df_productos.isnull().sum())
#Hay 2 valores nulos en categoria

#---------------------------------------------------------------------------#

#Buscamos valores duplicados en todas las columnas
#print(df_productos.duplicated().sum())
#Hay 1 valor duplicado

#----------------------------------------------------------------------------#

#Buscamos los productos con precios menores o iguales a 0
df_productos_precio0 = df_productos[df_productos["precio"] <= 0]
#Hay 2 productos con precio menor o igual a 0 (2,6)


#--------------------------------------------------------------------------#

#Creamos las mascaras para los registros que tienen categoria nula o vacia, precio menor o igual a 0 y los registros duplicados
mask_categoria_nula = df_productos["categoria"].isnull() | (df_productos["categoria"].astype(str).str.strip() == "")
mask_precio0_productos = df_productos["precio"] <= 0 | df_productos["precio"].isnull()
mask_duplicado = df_productos.duplicated(keep="first") | df_productos.duplicated(subset=["id_producto"], keep="first")

mask_rechazo = (
    mask_categoria_nula
    | mask_precio0_productos
    | mask_duplicado
)

motivos = pd.Series("", index=df_productos.index, dtype="string")
motivos.loc[mask_categoria_nula] = (
    motivos.loc[mask_categoria_nula] + "categoria_nula_o_vacia; "
)
motivos.loc[mask_precio0_productos] = (
    motivos.loc[mask_precio0_productos] + "precio_menor_o_igual_a_0; "
)
motivos.loc[mask_duplicado] = (
    motivos.loc[mask_duplicado] + "copia_exacta_duplicada; "
)   

df_rechazados_productos = df_productos.loc[mask_rechazo].copy()
df_productos_validos = df_productos.loc[~mask_rechazo].copy()
df_rechazados_productos["motivo_rechazo"] = (
    motivos.loc[mask_rechazo].str.rstrip("; ")
)


   id_producto              producto categoria  precio  \
0           13     Vino Malbec 750ml       NaN  5500.0   
1           13     Vino Malbec 750ml       NaN  5500.0   
2            2            Azúcar 1kg   Almacén -1500.0   
3            6  Arroz Largo Fino 1kg   Almacén     0.0   

                       motivo_rechazo  
0                      Categoria nula  
1  Categoria nula; registro duplicado  
2                Precio menor o igual  
3                Precio menor o igual  


In [ ]:


#Corregimos los tipos de datos de las columnas
df_ventas["fecha_convertida"] = pd.to_datetime(
    df_ventas["fecha"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

#Revisamos si hay fechas futuras en la columna fecha_convertida y vemos cuantas son
fechas_futuras = df_ventas[df_ventas["fecha_convertida"] > pd.Timestamp.now()]

#Revisamos si hay fechas futuras en la columna fecha y vemos cuantas son
raw = df_ventas["fecha"].astype("string").str.strip()

fecha_convertida = pd.Series(
    pd.NaT,
    index=df_ventas.index,
    dtype="datetime64[ns]"
)

formatos = [
    (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d"),  # 2026-11-12
    (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y"),  # 13/11/2026
    (r"\d{2}-\d{2}-\d{2}", "%d-%m-%y"),  # 09-10-26
]

for patron, formato in formatos:
    mascara = raw.str.fullmatch(patron, na=False)
    fecha_convertida.loc[mascara] = pd.to_datetime(
        raw.loc[mascara],
        format=formato,
        errors="coerce"
    )

df_ventas["fecha_convertida"] = fecha_convertida


#Hay 8 valores nulos en la columna fecha_convertida, lo que indica que hay 8 fechas que no se pudieron convertir correctamente al formato de fecha. Esto puede deberse a formatos de fecha inconsistentes o datos incorrectos en esas filas.
# (2020, 1905, 1514, 1305, 1039, 2338, 1145, 2491)


#--------------------------------------------------------------------------#


#Corregimos los tipos de datos de las columnas id_cliente, id_producto y cantidad
columnas = ["id_cliente", "id_producto", "cantidad"]
df_ventas[columnas] = df_ventas[columnas].apply(pd.to_numeric, errors="coerce")


#--------------------------------------------------------------------------#

#Buscamos cantidades menores o iguales a 0 en la columna cantidad
df_ventas_cantidad0 = df_ventas[df_ventas["cantidad"] <= 0]
print(len(df_ventas_cantidad0))
#Hay 18 valores menores o iguales a 0 en la columna cantidad


#--------------------------------------------------------------------------#
#Buscamos precios unitarios menores o iguales a 0 en la columna precio_unitario
df_ventas_precio0 = df_ventas[df_ventas["precio_unitario"] <= 0]
#print(len(df_ventas_precio0))
#Hay 8 valores menores o iguales a 0 en la columna precio_unitario


#--------------------------------------------------------------------------#

#Creamos las mascaras para los registros que tienen fecha nula o vacia, id_cliente nulo o vacio, id_producto nulo o vacio, precio_unitario menor o igual a 0, cantidad menor o igual a 0 y los registros duplicados
mask_fecha_nula = (
    df_ventas["fecha"].isnull() | (df_ventas["fecha"].astype(str).str.strip() == "") | df_ventas['fecha_convertida'].isna()
)
mask_id_cliente_nulo = df_ventas["id_cliente"].isnull() | (df_ventas["id_cliente"].astype(str).str.strip() == "") | df_ventas["id_cliente"].isin(df_clientes_validos["id_cliente"]) == False
mask_id_producto_nulo = df_ventas["id_producto"].isnull() | (df_ventas["id_producto"].astype(str).str.strip() == "") | df_ventas["id_producto"].isin(df_productos_validos["id_producto"]) == False
mask_precio0 = df_ventas["precio_unitario"] <= 0
mask_cantidad0 = df_ventas["cantidad"] <= 0
mask_duplicado = df_ventas.duplicated(keep="first") | df_ventas.duplicated(subset=["id_cliente", "id_producto", "fecha_convertida"], keep="first")

motivos = pd.Series("", index=df_ventas.index, dtype="string")
motivos.loc[mask_fecha_nula] = (
    motivos.loc[mask_fecha_nula] + "fecha_nula_o_mal_formateada; "
)
motivos.loc[mask_id_cliente_nulo] = (
    motivos.loc[mask_id_cliente_nulo] + "id_cliente_nulo_o_invalido; "
)
motivos.loc[mask_id_producto_nulo] = (
    motivos.loc[mask_id_producto_nulo] + "id_producto_nulo_o_invalido; "
)
motivos.loc[mask_precio0] = (
    motivos.loc[mask_precio0] + "precio_unitario_menor_o_igual_a_0; "
)
motivos.loc[mask_cantidad0] = (
    motivos.loc[mask_cantidad0] + "cantidad_menor_o_igual_a_0; "
)
motivos.loc[mask_duplicado] = (
    motivos.loc[mask_duplicado] + "copia_exacta_duplicada; "
)   

df_rechazados_ventas = df_ventas.loc[mask_rechazo].copy()
df_ventas_validos = df_ventas.loc[~mask_rechazo].copy()
df_rechazados_ventas["motivo_rechazo"] = (
    motivos.loc[mask_rechazo].str.rstrip("; ")
)


Valores nulos en fecha_convertida: 8
18
